In [1]:
df = spark.read.format("parquet") \
.option("header", "true") \
.option("inferSchema", "true") \
.load("abfss://sourceAndCleaning@onelake.dfs.fabric.microsoft.com/nycTaxiData.Lakehouse/Tables/dbo/greendata")
display(df)

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0fcf9b44-5019-4417-8fd2-9274192c2a30)

In [2]:
df.head()

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 4, Finished, Available, Finished, False)

Row(VendorID=2, lpep_pickup_datetime=datetime.datetime(2023, 1, 3, 18, 53, 13), lpep_dropoff_datetime=datetime.datetime(2023, 1, 3, 19, 0, 57), store_and_fwd_flag='N', RatecodeID=1.0, PULocationID=244, DOLocationID=220, passenger_count=1.0, trip_distance=4.23, fare_amount=18.4, extra=2.5, mta_tax=0.5, tip_amount=5.08, tolls_amount=3.0, ehail_fee=None, improvement_surcharge=1.0, total_amount=30.48, payment_type=1.0, trip_type=1.0, congestion_surcharge=0.0)

In [3]:
from pyspark.sql.functions import col

df_cleaned = df.drop("ehail_fee")
display(df_cleaned)

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9c26221f-4cda-4265-83dc-2fb22d7bc763)

In [4]:
df_cleaned.describe()

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 6, Finished, Available, Finished, False)

DataFrame[summary: string, VendorID: string, store_and_fwd_flag: string, RatecodeID: string, PULocationID: string, DOLocationID: string, passenger_count: string, trip_distance: string, fare_amount: string, extra: string, mta_tax: string, tip_amount: string, tolls_amount: string, improvement_surcharge: string, total_amount: string, payment_type: string, trip_type: string, congestion_surcharge: string]

In [5]:
df_cleaned = df_cleaned.filter(
    (col("trip_distance") > 0) &
    (col("fare_amount") > 0) &
    (col("passenger_count") > 0)
)

display(df_cleaned)

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6ba5c769-0331-4486-9765-8d445d2817ca)

In [6]:
df_cleaned = df_cleaned.na.fill({
    "congestion_surcharge" :0,
    "tip_amount": 0
})

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 8, Finished, Available, Finished, False)

In [7]:
df_cleaned = df_cleaned.withColumnRenamed("PULocationID", "Pickup_Location_ID") \
                        .withColumnRenamed("DOLocationID", "Dropoff_Location_ID")
display(df_cleaned)

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f345df4c-f8a2-4319-bfc4-9316d7381c85)

## ML Operations 

In [8]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.sql.functions import col
from pyspark.ml import Pipeline

feature_columns = ["trip_distance", "passenger_count", "RatecodeID", "congestion_surcharge", "tip_amount"]
target_column = "fare_amount"

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features_assembled")

scaler = StandardScaler(inputCol="features_assembled", outputCol="features_scaled", withMean=True, withStd=True)

pipeline = Pipeline(stages=[assembler, scaler])
prepared_data = pipeline.fit(df_cleaned).transform(df_cleaned)

prepared_data = prepared_data.select("features_scaled", col(target_column).alias("label"))

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 11, Finished, Available, Finished, False)

In [9]:
train_data, test_data = prepared_data.randomSplit([0.8,0.2], seed=42)

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 12, Finished, Available, Finished, False)

In [10]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

lr = LinearRegression(featuresCol="features_scaled", labelCol="label")
lr_model = lr.fit(train_data)

predictions = lr_model.transform(test_data)

evaluator = RegressionEvaluator(labelCol="label", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")

display(predictions.select("prediction", "label"))

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 13, Finished, Available, Finished, False)

Root Mean Squared Error (RMSE): 10.484712722913157


SynapseWidget(Synapse.DataFrame, aa874ab5-e126-4891-982a-a7e8287bef27)

In [11]:
from pyspark.sql.functions import corr

correlation = df_cleaned.select(corr("trip_distance", "fare_amount").alias("correlation")).collect()[0][0]
print(f"Correlation between trip_distance and fare_amount: {correlation}")

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 16, Finished, Available, Finished, False)

Correlation between trip_distance and fare_amount: 0.3386686432592058


In [12]:
from pyspark.sql.functions import when, mean, stddev, count

group_stats = df_cleaned.groupBy("payment_type").agg(
    mean("fare_amount").alias("mean_fare"),
    stddev("fare_amount").alias("stddev_fare"),
    count("fare_amount").alias("count")
).collect()

group1 = [row for row in group_stats if row["payment_type"] == 1][0]
group2 = [row for row in group_stats if row["payment_type"] == 2][0]

mean1, std1, n1 = group1["mean_fare"], group1["stddev_fare"], group1["count"]
mean2, std2, n2 = group2["mean_fare"], group2["stddev_fare"], group2["count"]

t_stat = (mean1 - mean2) / (((std1 ** 2 / n1) + (std2 ** 2 / n2)) ** 0.5)
print(f"T-statistic between payment_type 1 and 2: {t_stat}")

StatementMeta(, 3721b82d-79e5-4aae-84a5-86ebf4d16ea7, 17, Finished, Available, Finished, False)

T-statistic between payment_type 1 and 2: 2.185479495482603
